In [4]:
# ---- FANC notebook: imports + constants ----
import sys
import json
import pandas as pd

from caveclient import CAVEclient
import nglui.statebuilder as ngstbld

print("Python executable:", sys.executable)
print("Imports successful.")

DATASTACK = "fanc_production_mar2021"   # production FANC
# DATASTACK = "fanc_sandbox"            # switch to sandbox if you want

SPELUNKER_SITE = "https://spelunker.cave-explorer.org/"

Python executable: c:\Users\JHS\Documents\Python\cave_env\Scripts\python.exe
Imports successful.


In [ ]:
# ---- Discover accessible datastacks ----
##global_client = CAVEclient(global_only=True)

# This should return a list/dict of datastacks you can access
##ds = global_client.info.get_datastacks()
##print("Accessible datastacks:", ds)

Accessible datastacks: ['minnie35_public_v0', 'fanc_sandbox', 'flywire_fafb_sandbox', 'fanc_production_mar2021', 'mrgd', 'pinky_sandbox', 'minnie65_sandbox', 'flywire_fafb_public', 'minnie65_public']


In [6]:
# ---- FANC notebook: connect to CAVE ----
client = CAVEclient(DATASTACK)

print("Connected datastack:", DATASTACK)
print("Server:", client.server_address)
print("Image source:", client.info.image_source())
print("Seg source (raw):", client.info.segmentation_source())
print("Viewer resolution:", client.info.viewer_resolution())

# Convenience: segmentation source patched for Spelunker middleauth
SEG_SOURCE = client.info.segmentation_source().replace(
    "graphene://", "graphene://middleauth+"
)
print("Seg source (middleauth):", SEG_SOURCE)

Connected datastack: fanc_production_mar2021
Server: https://global.daf-apis.com
Image source: precomputed://gs://lee-lab_female-adult-nerve-cord/alignmentV4/em/rechunked
Seg source (raw): graphene://https://cave.fanc-fly.com/segmentation/table/mar2021_prod
Viewer resolution: [ 4.3  4.3 45. ]
Seg source (middleauth): graphene://middleauth+https://cave.fanc-fly.com/segmentation/table/mar2021_prod


In [8]:
# ---- FANC: create a base Spelunker viewer state (new nglui API) ----

from nglui.statebuilder import ViewerState

state = (
    ViewerState(
        dimensions=list(client.info.viewer_resolution()),
        infer_coordinates=False
    )
    .add_image_layer(
        source=client.info.image_source(),
        name="fanc"
    )
    .add_segmentation_layer(
        source=SEG_SOURCE,
        name="seg"
    )
)

state_dict = state.to_dict()

state_id = client.state.upload_state_json(state_dict)
url = client.state.build_neuroglancer_url(
    state_id,
    ngl_url=SPELUNKER_SITE
).replace("/?json_url=", "#!middleauth+")

print("Open this URL:")
print(url)

Open this URL:
https://spelunker.cave-explorer.org/#!middleauth+https://global.daf-apis.com/nglstate/api/v1/5573107694698496


In [9]:
#conferm the segmentation source that I am using:
print("Raw seg source:", client.info.segmentation_source())
print("Patched seg source:", SEG_SOURCE)

Raw seg source: graphene://https://cave.fanc-fly.com/segmentation/table/mar2021_prod
Patched seg source: graphene://middleauth+https://cave.fanc-fly.com/segmentation/table/mar2021_prod


In [ ]:
# this code forces to make a segmentation so i cna test if my URL shwos segmentation
state = (
    ViewerState(dimensions=list(client.info.viewer_resolution()), infer_coordinates=False)
    .add_image_layer(source=client.info.image_source(), name="fanc")
    .add_segmentation_layer(source=SEG_SOURCE, name="seg", segments=[1])
)

In [ ]:
# ---- Cell 2: Connect client + build viewer with a dummy segment ----Works!
client = CAVEclient(DATASTACK)

print("Connected datastack:", DATASTACK)
print("Server:", client.server_address)

# Sources
IMG_SOURCE = client.info.image_source()
RAW_SEG_SOURCE = client.info.segmentation_source()
SEG_SOURCE = RAW_SEG_SOURCE.replace("graphene://", "graphene://middleauth+")

print("Image source:", IMG_SOURCE)
print("Raw seg source:", RAW_SEG_SOURCE)
print("Patched seg source:", SEG_SOURCE)
print("Viewer resolution:", client.info.viewer_resolution())

# Build a minimal viewer state with a dummy segment selected
state = (
    ViewerState(dimensions=list(client.info.viewer_resolution()), infer_coordinates=False)
    .add_image_layer(source=IMG_SOURCE, name="fanc")
    .add_segmentation_layer(source=SEG_SOURCE, name="seg", segments=[1])  # <-- dummy segment
)

# Upload state + build Spelunker URL
state_id = client.state.upload_state_json(state.to_dict())
url = client.state.build_neuroglancer_url(state_id, ngl_url=SPELUNKER_SITE).replace(
    "/?json_url=", "#!middleauth+"
)

print("\nOpen this URL:")
print(url)

Connected datastack: fanc_production_mar2021
Server: https://global.daf-apis.com
Image source: precomputed://gs://lee-lab_female-adult-nerve-cord/alignmentV4/em/rechunked
Raw seg source: graphene://https://cave.fanc-fly.com/segmentation/table/mar2021_prod
Patched seg source: graphene://middleauth+https://cave.fanc-fly.com/segmentation/table/mar2021_prod
Viewer resolution: [ 4.3  4.3 45. ]

Open this URL:
https://spelunker.cave-explorer.org/#!middleauth+https://global.daf-apis.com/nglstate/api/v1/6563734607626240
